# ARC-AGI-3 Duck - STaR LoRA SFT (Qwen3.8-27B bf16)

Trains a LoRA adapter (r=16, a=32, dropout .05, all-6 linear targets) on
packed_sft.jsonl (3823 assistant turns distilled from our own 8.21-mean
winning runs + the public 1.25-model STaR set; SUBSAMPLE picks a quota-
sized subset deterministically). Output: PEFT adapter in
/kaggle/working/lora_adapter (~300MB), consumed by the anim-27b-lora
kernel via vLLM --enable-lora.

In [ ]:
import os, subprocess, sys

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
                       'peft==0.17.0', 'accelerate>=1.6', 'datasets>=3.0', 'transformers>=4.56.0'])
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

env = dict(os.environ)
env.update({
    'BASE_MODEL': '/kaggle/input/datasets/rahim3/qwen3-8-27b-bf16',
    'SFT_JSONL': '/kaggle/input/datasets/takumuhata/taaf-duck-sft-v1/packed_sft.jsonl',
    'OUT_DIR': '/kaggle/working/lora_adapter',
    'EPOCHS': '1',
    'SUBSAMPLE': '1200',
    'LR': '1e-4',
    'MAX_LEN': '8192',
    'HF_HUB_OFFLINE': '1',
    'TRANSFORMERS_OFFLINE': '1',
})
train = '/kaggle/input/datasets/takumuhata/taaf-duck-sft-v1/train_lora.py'
assert os.path.exists(train), train
assert os.path.exists(env['BASE_MODEL']), env['BASE_MODEL']
assert os.path.exists(env['SFT_JSONL']), env['SFT_JSONL']

proc = subprocess.run([sys.executable, train], env=env,
                      text=True, capture_output=True)
print(proc.stdout[-8000:])
print(proc.stderr[-4000:], file=sys.stderr)
assert proc.returncode == 0, 'train_lora failed'
import pathlib
files = sorted(p.name for p in pathlib.Path('/kaggle/working/lora_adapter').iterdir())
print('adapter files:', files)
print('done')
